In [ ]:
! pip install azure-ai-projects azure-monitor-opentelemetry opentelemetry-instrumentation-openai-v2

In [ ]:
import pkg_resources
print([d.project_name for d in pkg_resources.working_set if "openai" in d.project_name.lower()])


In [ ]:
! pip install --upgrade pip


In [ ]:
! pip install opentelemetry-instrumentation-openai-v2


In [ ]:
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

project_client = AIProjectClient(
     credential=DefaultAzureCredential(),
     endpoint="https://sarath-6690-resource.services.ai.azure.com/api/projects/sarath-6690",
)
connection_string = project_client.telemetry.get_application_insights_connection_string()

In [ ]:
! /workspaces/Azure-ai-foundry01/.venv/bin/python -m ensurepip --upgrade
! /workspaces/Azure-ai-foundry01/.venv/bin/python -m pip install --upgrade pip
! /workspaces/Azure-ai-foundry01/.venv/bin/python -m pip install opentelemetry-instrumentation-openai-v2
# Restart is needed


In [ ]:
from azure.monitor.opentelemetry import configure_azure_monitor
from opentelemetry.instrumentation.openai_v2 import OpenAIInstrumentor

configure_azure_monitor(connection_string=connection_string)
OpenAIInstrumentor().instrument()

In [ ]:
client = project_client.get_openai_client()

response = client.chat.completions.create(
    model="gpt-4o", 
    messages=[{"role": "user", "content": "Write a short poem on open telemetry."}],
)
print(response.choices[0].message.content)

In [ ]:
from opentelemetry import trace

tracer = trace.get_tracer(__name__)

In [ ]:
def build_prompt_with_context(claim: str, context: str) -> str:
    return [{'role': 'system', 'content': "I will ask you to assess whether a particular scientific claim, based on evidence provided. Output only the text 'True' if the claim is true, 'False' if the claim is false, or 'NEE' if there's not enough evidence."},
            {'role': 'user', 'content': f"""
                The evidence is the following: {context}

                Assess the following claim on the basis of the evidence. Output only the text 'True' if the claim is true, 'False' if the claim is false, or 'NEE' if there's not enough evidence. Do not output any other text.

                Claim:
                {claim}

                Assessment:
            """}]

@tracer.start_as_current_span("assess_claims_with_context")
def assess_claims_with_context(claims, contexts):
    responses = []
    for claim, context in zip(claims, contexts):
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=build_prompt_with_context(claim=claim, context=context),
        )
        responses.append(response.choices[0].message.content.strip('., '))

    return responses

In [ ]:
claims = [
    "Drinking water improves memory.",
    "The Earth is flat.",
    "Vitamin C prevents the common cold."
]

contexts = [
    "A recent study showed that hydration improves cognitive performance.",
    "Scientific consensus and satellite imagery confirm the Earth is spherical.",
    "Multiple clinical trials have found no significant effect of Vitamin C in preventing colds."
]

# Call your function
assessments = assess_claims_with_context(claims, contexts)

print(assessments)
